# 02. Joint whitelist, tokens, and uniform masks

We will build the exact model input, verify the deduplicated joint list, and check that masking never depends on movement intensity.

## Run this notebook independently

From `experiments/aaai`, run:

```bash
uv sync
uv run jupyter lab
```

Select the kernel from this environment. Python 3.12 and all notebook libraries are declared in `pyproject.toml`. Every notebook repeats its setup and loads its own inputs. No variable from another notebook is required.

A fresh installation defaults to `AAAI_DEMO=1` for a small synthetic software check. This prepared workspace uses `AAAI_DEMO=0` after real-data ingestion and validation. Its condition parameters are invented and its scores are not research evidence. Set `AAAI_DEMO=0` in the repository root `.env` after the ingestion workflow has produced a complete pose corpus. All notebooks use the same root `.env`. No API key is needed for JEPA or Random Forest training. Restart the kernel after changing configuration.

Only the root environment file contains credentials. `load_env` and `find_env` below are aliases for the real `python-dotenv` functions, `load_dotenv` and `find_dotenv`.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys
# Find the repository from either its root or this notebook directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ambient').is_dir() and (p / 'data/gavd').is_dir())
BASE = ROOT / 'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())  # requested spelling, using aliases for python-dotenv's real API
os.chdir(BASE)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.settings import Settings
from aaai_gait.data import build_manifest, grouped_split, LABELS
from aaai_gait.media import show_video_context, plot_motion
settings = Settings()
torch.set_num_threads(2)
manifest = grouped_split(build_manifest(ROOT), seed=settings.seed)
print('Mode:', 'SYNTHETIC SOFTWARE DEMONSTRATION' if settings.demo else 'GAVD RESEARCH RUN')
print('Current annotation sequences:', len(manifest), 'unique sources:', manifest.source_id.nunique())

## Connect the experiment to walking

Watch a short annotated normal training sequence when local media is available. The preview is an observed video. Synthetic arrays used below are a separate software demonstration.

In [ ]:
video_row = show_video_context(manifest, frame_number_base=settings.frame_number_base)

# Deduplicated joint whitelist

The selected features use the following 16 distinct BlazePose joints. The table is in numerical order. The model receives only these joints, so its uniform masking sampler cannot mask any other landmark. Original BlazePose indices and compact model positions are different: compact position 0 means BlazePose joint 11.

|BLAZEPOSE_33 index|Keypoint name|Features involved|
|---|---|---|
|11|Left shoulder|Left hip angle; torso scale and orientation; learned shoulder trajectory|
|12|Right shoulder|Right hip angle; torso scale and orientation; learned shoulder trajectory|
|13|Left elbow|Learned arm trajectory and shoulder-elbow-wrist relationship|
|14|Right elbow|Learned arm trajectory and shoulder-elbow-wrist relationship|
|15|Left wrist|Wrist speed, variability and excursion; learned arm swing|
|16|Right wrist|Wrist speed, variability and excursion; learned arm swing|
|23|Left hip|Pelvis center; torso scale; left hip and knee angles; bilateral angle asymmetry|
|24|Right hip|Pelvis center; torso scale; right hip and knee angles; bilateral angle asymmetry|
|25|Left knee|Left hip, knee and ankle angles; bilateral angle asymmetry; learned leg trajectory|
|26|Right knee|Right hip, knee and ankle angles; bilateral angle asymmetry; learned leg trajectory|
|27|Left ankle|Left knee and ankle angles; speed and excursion; ankle separation|
|28|Right ankle|Right knee and ankle angles; speed and excursion; ankle separation|
|29|Left heel|Learned foot orientation and heel trajectory|
|30|Right heel|Learned foot orientation and heel trajectory|
|31|Left foot index|Left ankle angle; learned foot orientation and toe trajectory|
|32|Right foot index|Right ankle angle; learned foot orientation and toe trajectory|

The classical 36-feature baseline does not use every selected coordinate independently. Elbows and heels contribute to learned trajectories in JEPA. Their inclusion is an explicit anatomical design choice. Face and finger-tip landmarks are excluded. The user did not provide a separate list of feature-to-joint definitions, so this table declares the experimental subset rather than claiming it is uniquely required by S-JEPA or clinically optimal.

Bilateral angle differences summarize image-plane asymmetry. They do not identify the affected side or measure three-dimensional joint mechanics. Keep left/right identities, avoid horizontal reflection augmentation in the main analysis, and do not interpret normalized 2D distances as physical stride length.

![Selected joints](images/joint_whitelist.svg)

In [ ]:
from aaai_gait.workflow import prepare_corpus
x, meta, provenance = prepare_corpus(settings)
print('Window tensor:', x.shape, '[windows, frames, selected joints, XY]')
display(meta.groupby(['split','label']).agg(windows=('sequence_id','size'), sequences=('sequence_id','nunique'), sources=('source_id','nunique')))

In [ ]:
from aaai_gait.model import JOINT_INDICES, ModelConfig, SJEPA, uniform_mask
from aaai_gait.data import JOINTS
assert tuple(JOINTS)==JOINT_INDICES
assert list(JOINT_INDICES)==sorted(set(JOINT_INDICES))
display(pd.DataFrame({'compact_position':range(16),'BLAZEPOSE_33 index':JOINT_INDICES}))
plot_motion(x[0],title='Synthetic software check' if settings.demo else 'Retained GAVD motion');plt.show()

## 1. Build a temporal patch for one joint

The input tensor has shape `[B, 32, 16, 2]`. Four consecutive XY pairs for the same joint become eight numbers. A learned linear map converts those eight numbers into a 64-dimensional token. Eight time patches times 16 joints gives 128 tokens. Learned temporal and joint embeddings retain identity.

![Tokens](images/tokenization.svg)

In [ ]:
cfg=ModelConfig()
model=SJEPA(cfg)
batch=torch.as_tensor(x[:4])
tokens=model.view_encoder.tokenize(batch)
print('Input:',tuple(batch.shape),'tokens:',tuple(tokens.shape))
assert tokens.shape==(4,128,64)

## 2. Sample without reading the skeleton

`uniform_mask` accepts batch size, token count, ratio, and a random generator. It accepts no coordinates, label, velocity, or motion-intensity argument. At a 0.5 ratio, 64 positions per example are hidden without replacement. Every permitted joint-time token has the same marginal selection probability.

Masking a token hides four frames of one joint from the context encoder. The complete target encoder sees all observed tokens in the clean sequence. Missing patches remain invalid and cannot become target labels. A joint can be entirely hidden in one draw, but no joint is permanently reserved as context.

![Uniform masks](images/uniform_mask.svg)

In [ ]:
generator=torch.Generator().manual_seed(settings.seed)
masks=uniform_mask(2000,cfg.tokens,.5,generator=generator)
print('Hidden count per example:',masks.sum(1).unique().tolist())
frequency=masks.float().mean(0).reshape(cfg.time_tokens,cfg.joints).numpy()
plt.figure(figsize=(10,3));plt.imshow(frequency,aspect='auto',vmin=.4,vmax=.6,cmap='viridis')
plt.xticks(range(16),JOINT_INDICES);plt.xlabel('BlazePose joint index');plt.ylabel('Time patch');plt.colorbar(label='Empirical hidden fraction');plt.title('Uniform-mask software check');plt.show()

## 3. Understand positional stochasticity

Our optional StoP-inspired mode adds Gaussian noise to hidden predictor queries through the same learned projection that maps context features. It does not shuffle joint identities or perturb teacher coordinates. The default model uses deterministic learned positions. Inference switches positional noise off.

This is an ablation to evaluate, not an assumed improvement. Compare identical splits and seeds with `spe=False` and `spe=True`. Read the [literature review](docs/literature_review.md) before describing it as the original StoP implementation.

In [ ]:
spe_model=SJEPA(ModelConfig(spe=True))
mask=uniform_mask(len(batch),cfg.tokens,generator=torch.Generator().manual_seed(9))
spe_model.eval()
a=spe_model.embed(batch);b=spe_model.embed(batch)
assert torch.equal(a,b)
print('Evaluation embeddings are deterministic.')

## What to record before a paper run

Save the current annotation hash, source assignments, retained windows, exclusions, software versions, model configuration, seed, and exact command. Keep synthetic artifacts separate. Revisit the video when an apparent anomaly could instead be a crop error, occlusion, or viewpoint change.

The [research review](docs/literature_review.md) explains source methods and adaptations. The [paper plan](docs/paper_plan.md) defines claims, ablations, and limits.